# A4 cross-dataset, four directions — symmetric transfer benchmark

Experiment 13 covered PhysioNet -> IV-2a (AUC 0.694). This notebook runs the three missing directions plus a fourth (IV-2a -> Lee 2019) so the cross-dataset transfer claim is no longer one-directional:

    1. IV-2a -> PhysioNet         (9-subject train, 104 unseen)
    2. PhysioNet -> Lee 2019      (80-subject train, 54 unseen)
    3. Lee 2019 -> PhysioNet      (40-subject train, 104 unseen)
    4. IV-2a -> Lee 2019          (9-subject train, 54 unseen)

Prerequisites:
  - Run `A3_lee2019_download.ipynb` first (compact cache must be 108/108).
  - PhysioNet imagery is fetched via the in-repo prefetcher inside the run cell.
  - IV-2a is fetched via moabb on demand.

**Runtime → L4 GPU.** Expected wall: ~75-110 min for all four directions.

**Don't `Save a copy in GitHub` from Colab.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
os.environ['BCI_LEE2019_CACHE'] = '/content/drive/MyDrive/bci_cache'
import glob, sys
WIN = '/content/drive/MyDrive/bci_cache/lee2019_mi/windowed'
files = sorted(glob.glob(os.path.join(WIN, '*.npz')))
print(f'Lee 2019 cached: {len(files)}/108')
if len(files) < 108:
    sys.exit('!!! Lee 2019 cache incomplete; run A3_lee2019_download.ipynb first.')

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

## Direction 1: IV-2a -> PhysioNet (9-subject train, 104 unseen)

In [ ]:
!BCI_LEE2019_CACHE=/content/drive/MyDrive/bci_cache PYTHONUNBUFFERED=1 \
  python -u -m experiments.26_a4_cross_dataset_symmetric \
  --direction iv2a_to_physionet --all

## Direction 2: PhysioNet -> Lee 2019 (80-subject train, 54 unseen)

In [ ]:
!BCI_LEE2019_CACHE=/content/drive/MyDrive/bci_cache PYTHONUNBUFFERED=1 \
  python -u -m experiments.26_a4_cross_dataset_symmetric \
  --direction physionet_to_lee2019 --all

## Direction 3: Lee 2019 -> PhysioNet (40-subject train, 104 unseen)

In [ ]:
!BCI_LEE2019_CACHE=/content/drive/MyDrive/bci_cache PYTHONUNBUFFERED=1 \
  python -u -m experiments.26_a4_cross_dataset_symmetric \
  --direction lee2019_to_physionet --all

## Direction 4: IV-2a -> Lee 2019 (9-subject train, 54 unseen)

In [ ]:
!BCI_LEE2019_CACHE=/content/drive/MyDrive/bci_cache PYTHONUNBUFFERED=1 \
  python -u -m experiments.26_a4_cross_dataset_symmetric \
  --direction iv2a_to_lee2019 --all

## Paste-back: emit all four result JSONs + run metadata

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
TAG = 'a4_xds'
try:
    git_sha = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
except Exception:
    git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
directions = ['iv2a_to_physionet','physionet_to_lee2019','lee2019_to_physionet','iv2a_to_lee2019']
result_paths = [f'results/26_a4_xds_{d}.json' for d in directions]
meta = {'run_id': run_id, 'experiment': 'experiments.26_a4_cross_dataset_symmetric',
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': result_paths}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json','w') as f: json.dump(meta, f, indent=2)
for path in result_paths:
    print(f'--- BEGIN {os.path.basename(path)} ---')
    if os.path.exists(path):
        print(open(path).read())
    else:
        print(f'MISSING: {path}')
    print(f'--- END {os.path.basename(path)} ---')
    print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')